In [5]:
import evidently

from evidently import Report
from evidently.presets import DataDriftPreset
from evidently.metrics import ValueDrift
import pandas as pd

from src.constants import DATASET_DIR, OUTPUT_DIR

In [2]:
reference_df = pd.read_parquet(DATASET_DIR /"reference.parquet")
current_df = pd.read_parquet(DATASET_DIR / "current.parquet")

assert set(reference_df.columns) == set(current_df.columns), "Reference and current datasets must have the same columns."

target_cols = [col for col in reference_df.columns if "target" in col.lower()]
feature_cols = [col for col in reference_df.columns if col not in target_cols]

target_cols = [col for col in reference_df.columns if "target" in col.lower()]
feature_cols = [col for col in reference_df.columns if col not in target_cols]

threshold = 0.2


In [3]:
report = Report(metrics=[
    DataDriftPreset(drift_share=threshold), # Data Drift
    *[ValueDrift(column=col) for col in feature_cols] # Concept Draft
])

In [4]:
report_run = report.run(reference_data=reference_df, current_data=current_df)

In [10]:
OUTPUT_DIR / f"data_drift_report.html"

PosixPath('/home/ec2-user/aqi_probability_prediction/reports/data_drift_report.html')

In [14]:
report_run.save_html(str(OUTPUT_DIR / f"data_drift_report.html"))

In [15]:

report_dict = report_run.dict()


In [17]:
drift_detected = report_dict["metrics"][0]["value"]["share"] > threshold

# Extract feature-level drift scores from ValueDrift metrics
feature_drifts = {}
for metric in report_dict["metrics"]:
    if "ValueDrift(column=" in metric["metric_id"]:
        # Extract column name from metric_id (e.g., "ValueDrift(column=f0)" -> "f0")
        column_name = metric["metric_id"].split("column=")[1].rstrip(")")
        drift_score = float(metric["value"])
        feature_drifts[column_name] = drift_score

# Select up to 3 features with highest drift scores
sorted_features = sorted(feature_drifts.items(), key=lambda x: x[1], reverse=True)
all_features = dict(sorted_features)
selected_features = dict(sorted_features[:3])

overall_drift_score = sum(all_features.values()) / len(all_features) if all_features else 0

output = {
    "drift_detected": drift_detected,
    "feature_drifts": selected_features,
    "overall_drift_score": overall_drift_score,
}

In [19]:
with open(OUTPUT_DIR / "drift_report.json", "w") as f:
    import json
    json.dump(output, f, indent=4)
    